## Scraping avec BeautifulSoup
Source : https://quotes.toscrape.com/ 

In [11]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

## 1. Requête HTTP

In [12]:
url = "https://quotes.toscrape.com/"
HEADERS = {'User-Agent': 'Mozilla/5.0'}

page = requests.get(url, headers=HEADERS)
print("Statut HTTP :", page.status_code) 

Statut HTTP : 200


## 2. Parser le HTML avec BeautifulSoup

In [13]:
soup = BeautifulSoup(page.content, 'html.parser')
print(soup.prettify()[:500])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scrape
      </a>
     </h1>
    </div>
    <div class="col-md-4">
     <p>
      <a href="/login">
   


## 3. Extraire les données

In [14]:
quote_blocks = soup.find_all('div', class_='quote')
print(f"{len(quote_blocks)} citations trouvées")

10 citations trouvées


In [15]:
quotes = []
authors = []
tags_list = []

for block in quote_blocks:

    quote_text = block.find('span', class_='text').get_text(strip=True)
    
    author = block.find('small', class_='author').get_text(strip=True)
    
    tags = [tag.get_text(strip=True) for tag in block.find_all('a', class_='tag')]
    tags_str = ', '.join(tags)
    
    quotes.append(quote_text)
    authors.append(author)
    tags_list.append(tags_str)

print("Exemple :")
print("Citation :", quotes[0])
print("Auteur   :", authors[0])
print("Tags     :", tags_list[0])

Exemple :
Citation : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Auteur   : Albert Einstein
Tags     : change, deep-thoughts, thinking, world


## 4. Stocker dans un DataFrame

In [16]:
data = pd.DataFrame({
    'Quote': quotes,
    'Author': authors,
    'Tags': tags_list
})

print(data.shape)
data.head()

(10, 3)


,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


## 5. Scraper toutes les pages

In [17]:
all_quotes = []
all_authors = []
all_tags = []

base_url = "https://quotes.toscrape.com/page/{}/"
page_num = 1

while True:
    url = base_url.format(page_num)
    page = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(page.content, 'html.parser')
    
    blocks = soup.find_all('div', class_='quote')
    
    if not blocks:
        break
    
    for block in blocks:
        all_quotes.append(block.find('span', class_='text').get_text(strip=True))
        all_authors.append(block.find('small', class_='author').get_text(strip=True))
        tags = [t.get_text(strip=True) for t in block.find_all('a', class_='tag')]
        all_tags.append(', '.join(tags))
    
    print(f"Page {page_num} : {len(blocks)} citations")
    page_num += 1

print(f"\nTotal : {len(all_quotes)} citations collectées")

Page 1 : 10 citations
Page 2 : 10 citations
Page 3 : 10 citations
Page 4 : 10 citations
Page 5 : 10 citations
Page 6 : 10 citations
Page 7 : 10 citations
Page 8 : 10 citations
Page 9 : 10 citations
Page 10 : 10 citations

Total : 100 citations collectées


In [18]:
df_full = pd.DataFrame({
    'Quote': all_quotes,
    'Author': all_authors,
    'Tags': all_tags
})

print(df_full.shape)
df_full.head(10)

(100, 3)


,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile"


## 6. Sauvegarder en CSV

In [19]:
df_full.to_csv('quotes.csv', index=False)
print("Fichier sauvegardé : quotes.csv")

Fichier sauvegardé : quotes.csv
